# Week 1: Linear Regression, Polynomial Terms, Interactions, and Multicollinearity

This capstone project focuses on predicting chronic disease outcomes using healthcare datasets. The objective is to compare how well different datasets can be modeled and to better understand how the characteristics of each dataset influence predictive performance.

In this notebook, linear regression is used as a starting point to explore relationships between the features and target variables in the diabetes and Alzheimer's datasets. Baseline linear regression models are developed and then extended with polynomial and interaction terms. Variance Inflation Factor (VIF) is also used to evaluate multicollinearity.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

## 1. Load the Datasets

The diabetes and Alzheimer's datasets are loaded into Pandas DataFrames. The shape and first few rows are displayed to confirm that the data loaded correctly.

In [2]:
diabetes = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")
alzheimers = pd.read_csv("alzheimers_disease_data.csv")

print("Diabetes Dataset Shape:")
print(diabetes.shape)
display(diabetes.head())

print("Alzheimer's Dataset Shape:")
print(alzheimers.shape)
display(alzheimers.head())

Diabetes Dataset Shape:
(253680, 22)


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


Alzheimer's Dataset Shape:
(2149, 35)


,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid


## 2. Prepare the Data

The target variable is separated from the predictor variables for each dataset. The diabetes dataset uses **Diabetes_012** as the target variable, while the Alzheimer's dataset uses **Diagnosis**. Non-predictive fields, such as patient identifiers and administrative information, are removed from the Alzheimer's dataset before modeling.

In [3]:
# Diabetes
X_diabetes = diabetes.drop(columns=["Diabetes_012"])
y_diabetes = diabetes["Diabetes_012"]

# Alzheimer's
X_alzheimers = alzheimers.drop(
    columns=["PatientID", "DoctorInCharge", "Diagnosis"]
)
y_alzheimers = alzheimers["Diagnosis"]

print("Diabetes Features:", X_diabetes.shape)
print("Diabetes Target:", y_diabetes.shape)

print()

print("Alzheimer's Features:", X_alzheimers.shape)
print("Alzheimer's Target:", y_alzheimers.shape)

Diabetes Features: (253680, 21)
Diabetes Target: (253680,)

Alzheimer's Features: (2149, 32)
Alzheimer's Target: (2149,)


## 3. Baseline Linear Regression

A baseline linear regression model is fit to each dataset using the original predictor variables. The data is first divided into training and testing sets using an 80/20 split. Model performance is then evaluated using the coefficient of determination (R²) and the Root Mean Squared Error (RMSE). These results serve as a baseline for comparison with the polynomial and interaction models developed later in this notebook.

In [4]:
# Split the diabetes dataset
X_train_diabetes, X_test_diabetes, y_train_diabetes, y_test_diabetes = train_test_split(
    X_diabetes,
    y_diabetes,
    test_size=0.20,
    random_state=42
)

# Split the Alzheimer's dataset
X_train_alz, X_test_alz, y_train_alz, y_test_alz = train_test_split(
    X_alzheimers,
    y_alzheimers,
    test_size=0.20,
    random_state=42
)

In [5]:
# Diabetes Linear Regression
diabetes_lr = LinearRegression()
diabetes_lr.fit(X_train_diabetes, y_train_diabetes)

diabetes_preds = diabetes_lr.predict(X_test_diabetes)

diabetes_r2 = r2_score(y_test_diabetes, diabetes_preds)
diabetes_rmse = np.sqrt(mean_squared_error(y_test_diabetes, diabetes_preds))

print("Diabetes")
print(f"R²: {diabetes_r2:.4f}")
print(f"RMSE: {diabetes_rmse:.4f}")

print()

# Alzheimer's Linear Regression
alz_lr = LinearRegression()
alz_lr.fit(X_train_alz, y_train_alz)

alz_preds = alz_lr.predict(X_test_alz)

alz_r2 = r2_score(y_test_alz, alz_preds)
alz_rmse = np.sqrt(mean_squared_error(y_test_alz, alz_preds))

print("Alzheimer's")
print(f"R²: {alz_r2:.4f}")
print(f"RMSE: {alz_rmse:.4f}")

Diabetes
R²: 0.1733
RMSE: 0.6323

Alzheimer's
R²: 0.4082
RMSE: 0.3683


### Baseline Linear Regression Results

The baseline linear regression model achieved an R² value of approximately 0.173 for the diabetes dataset and 0.408 for the Alzheimer's dataset. These results indicate that the Alzheimer's dataset contains stronger linear relationships between the predictor variables and the target than the diabetes dataset.

These baseline models provide a reference point for evaluating whether polynomial and interaction terms improve predictive performance.

## 4. Polynomial Regression

Polynomial regression extends the baseline linear regression model by adding squared terms for selected continuous variables. This allows the model to capture nonlinear relationships that may exist between the predictors and the target variables. The resulting models are evaluated using the same metrics as the baseline model to determine whether the additional complexity improves predictive performance.

In [6]:
# Diabetes polynomial features
X_train_diabetes_poly = X_train_diabetes.copy()
X_test_diabetes_poly = X_test_diabetes.copy()

poly_features_diabetes = [
    "BMI",
    "Age",
    "GenHlth",
    "MentHlth",
    "PhysHlth"
]

for feature in poly_features_diabetes:
    X_train_diabetes_poly[f"{feature}_Squared"] = X_train_diabetes_poly[feature] ** 2
    X_test_diabetes_poly[f"{feature}_Squared"] = X_test_diabetes_poly[feature] ** 2

# Alzheimer's polynomial features
X_train_alz_poly = X_train_alz.copy()
X_test_alz_poly = X_test_alz.copy()

poly_features_alz = [
    "Age",
    "BMI",
    "AlcoholConsumption",
    "PhysicalActivity",
    "DietQuality",
    "SleepQuality",
    "SystolicBP",
    "DiastolicBP",
    "CholesterolTotal",
    "CholesterolLDL",
    "CholesterolHDL",
    "CholesterolTriglycerides",
    "MMSE",
    "FunctionalAssessment",
    "ADL"
]

for feature in poly_features_alz:
    X_train_alz_poly[f"{feature}_Squared"] = X_train_alz_poly[feature] ** 2
    X_test_alz_poly[f"{feature}_Squared"] = X_test_alz_poly[feature] ** 2

In [7]:
# Diabetes Polynomial Regression
diabetes_poly = LinearRegression()
diabetes_poly.fit(X_train_diabetes_poly, y_train_diabetes)

diabetes_poly_preds = diabetes_poly.predict(X_test_diabetes_poly)

diabetes_poly_r2 = r2_score(y_test_diabetes, diabetes_poly_preds)
diabetes_poly_rmse = np.sqrt(mean_squared_error(y_test_diabetes, diabetes_poly_preds))

print("Diabetes")
print(f"R²: {diabetes_poly_r2:.4f}")
print(f"RMSE: {diabetes_poly_rmse:.4f}")

print()

# Alzheimer's Polynomial Regression
alz_poly = LinearRegression()
alz_poly.fit(X_train_alz_poly, y_train_alz)

alz_poly_preds = alz_poly.predict(X_test_alz_poly)

alz_poly_r2 = r2_score(y_test_alz, alz_poly_preds)
alz_poly_rmse = np.sqrt(mean_squared_error(y_test_alz, alz_poly_preds))

print("Alzheimer's")
print(f"R²: {alz_poly_r2:.4f}")
print(f"RMSE: {alz_poly_rmse:.4f}")

Diabetes
R²: 0.1778
RMSE: 0.6305

Alzheimer's
R²: 0.4261
RMSE: 0.3627


### Polynomial Regression Results

Adding polynomial terms resulted in modest improvements for both datasets. The diabetes model achieved a slight increase in R² and a small reduction in RMSE, suggesting that nonlinear relationships provide limited additional predictive information.

The Alzheimer's dataset showed a larger improvement in both R² and RMSE, indicating that polynomial terms were better able to capture nonlinear relationships between the predictor variables and the target. Overall, these results suggest that incorporating nonlinear features can improve model performance, particularly for the Alzheimer's dataset.

## 5. Interaction Terms

Interaction terms allow the effect of one predictor variable to depend on another. Selected interaction terms were added to each dataset based on relationships that are reasonable in a healthcare context. The resulting models were then evaluated to determine whether these interactions improve predictive performance.

In [8]:
# Diabetes interaction features
X_train_diabetes_inter = X_train_diabetes.copy()
X_test_diabetes_inter = X_test_diabetes.copy()

X_train_diabetes_inter["BMI_Age"] = (
    X_train_diabetes_inter["BMI"] * X_train_diabetes_inter["Age"]
)
X_test_diabetes_inter["BMI_Age"] = (
    X_test_diabetes_inter["BMI"] * X_test_diabetes_inter["Age"]
)

X_train_diabetes_inter["GenHlth_PhysHlth"] = (
    X_train_diabetes_inter["GenHlth"] * X_train_diabetes_inter["PhysHlth"]
)
X_test_diabetes_inter["GenHlth_PhysHlth"] = (
    X_test_diabetes_inter["GenHlth"] * X_test_diabetes_inter["PhysHlth"]
)

X_train_diabetes_inter["HighBP_HighChol"] = (
    X_train_diabetes_inter["HighBP"] * X_train_diabetes_inter["HighChol"]
)
X_test_diabetes_inter["HighBP_HighChol"] = (
    X_test_diabetes_inter["HighBP"] * X_test_diabetes_inter["HighChol"]
)

# Alzheimer's interaction features
X_train_alz_inter = X_train_alz.copy()
X_test_alz_inter = X_test_alz.copy()

X_train_alz_inter["Age_MMSE"] = (
    X_train_alz_inter["Age"] * X_train_alz_inter["MMSE"]
)
X_test_alz_inter["Age_MMSE"] = (
    X_test_alz_inter["Age"] * X_test_alz_inter["MMSE"]
)

X_train_alz_inter["MMSE_FunctionalAssessment"] = (
    X_train_alz_inter["MMSE"] * X_train_alz_inter["FunctionalAssessment"]
)
X_test_alz_inter["MMSE_FunctionalAssessment"] = (
    X_test_alz_inter["MMSE"] * X_test_alz_inter["FunctionalAssessment"]
)

X_train_alz_inter["Age_FunctionalAssessment"] = (
    X_train_alz_inter["Age"] * X_train_alz_inter["FunctionalAssessment"]
)
X_test_alz_inter["Age_FunctionalAssessment"] = (
    X_test_alz_inter["Age"] * X_test_alz_inter["FunctionalAssessment"]
)

In [9]:
# Diabetes Interaction Model
diabetes_inter = LinearRegression()
diabetes_inter.fit(X_train_diabetes_inter, y_train_diabetes)

diabetes_inter_preds = diabetes_inter.predict(X_test_diabetes_inter)

diabetes_inter_r2 = r2_score(y_test_diabetes, diabetes_inter_preds)
diabetes_inter_rmse = np.sqrt(mean_squared_error(y_test_diabetes, diabetes_inter_preds))

print("Diabetes")
print(f"R²: {diabetes_inter_r2:.4f}")
print(f"RMSE: {diabetes_inter_rmse:.4f}")

print()

# Alzheimer's Interaction Model
alz_inter = LinearRegression()
alz_inter.fit(X_train_alz_inter, y_train_alz)

alz_inter_preds = alz_inter.predict(X_test_alz_inter)

alz_inter_r2 = r2_score(y_test_alz, alz_inter_preds)
alz_inter_rmse = np.sqrt(mean_squared_error(y_test_alz, alz_inter_preds))

print("Alzheimer's")
print(f"R²: {alz_inter_r2:.4f}")
print(f"RMSE: {alz_inter_rmse:.4f}")

Diabetes
R²: 0.1841
RMSE: 0.6281

Alzheimer's
R²: 0.4260
RMSE: 0.3627


### Interaction Model Results

Adding interaction terms improved the diabetes model, producing the highest R² value and the lowest RMSE among the three regression approaches. This suggests that certain health indicators work together to better explain diabetes status than when considered independently.

For the Alzheimer's dataset, the interaction model produced results that were nearly identical to the polynomial model. This indicates that the selected interaction terms contributed little additional predictive information beyond the nonlinear relationships already captured by the polynomial features.

## 6. Variance Inflation Factor (VIF)

Polynomial features can introduce multicollinearity because squared terms are often correlated with their original variables. The Variance Inflation Factor (VIF) is used to evaluate the extent of multicollinearity in each dataset. Higher VIF values indicate stronger relationships among predictor variables and less stable coefficient estimates.

In [10]:
# Function to calculate VIF
def calculate_vif(X):

    X = sm.add_constant(X)

    vif = pd.DataFrame()
    vif["Feature"] = X.columns
    vif["VIF"] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]

    return vif

In [11]:
diabetes_vif = calculate_vif(
    X_train_diabetes_poly.sample(5000, random_state=42)
)

diabetes_vif

,Feature,VIF
0,const,218.424199
1,HighBP,1.319153
2,HighChol,1.179669
3,CholCheck,1.029198
4,BMI,15.710915
5,Smoker,1.105571
6,Stroke,1.083508
7,HeartDiseaseorAttack,1.169520
8,PhysActivity,1.153018
9,Fruits,1.121103


In [12]:
alz_vif = calculate_vif(X_train_alz_poly)

alz_vif

,Feature,VIF
0,const,9625.127735
1,Age,364.562098
2,Gender,1.020048
3,Ethnicity,1.022613
4,EducationLevel,1.026135
5,BMI,72.851789
6,Smoking,1.025681
7,AlcoholConsumption,16.604017
8,PhysicalActivity,16.152889
9,DietQuality,16.670067


### Variance Inflation Factor Results

The VIF analysis shows that many of the original predictor variables have relatively low VIF values, indicating little multicollinearity. However, after adding polynomial features, several variables exhibit substantially higher VIF values. This is expected because squared terms are naturally correlated with their corresponding original variables.

The diabetes dataset shows moderate increases in VIF for variables such as BMI, Age, and General Health, while the Alzheimer's dataset exhibits much larger VIF values for several continuous variables. These results demonstrate that polynomial features can improve predictive performance but also increase multicollinearity, creating a tradeoff between model complexity and coefficient stability.

## Conclusions

This analysis compared baseline linear regression, polynomial regression, and interaction models using the diabetes and Alzheimer's datasets.

For the diabetes dataset, the interaction model achieved the best performance, increasing the R² value from 0.1733 to 0.1841 while reducing the RMSE. This suggests that interactions between selected health indicators provided additional predictive information beyond the baseline model.

For the Alzheimer's dataset, polynomial regression produced the highest R² value (0.4261), although the interaction model achieved nearly identical performance. This indicates that nonlinear relationships contributed more to predictive performance than the selected interaction terms.

The VIF analysis showed that adding polynomial features increased multicollinearity, particularly between continuous variables and their squared terms. This illustrates the tradeoff between improving predictive performance and maintaining stable model coefficients.

Overall, polynomial and interaction terms provided modest improvements over the baseline linear regression models, with the degree of improvement varying across the two healthcare datasets.

In [13]:
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Polynomial",
        "Interaction"
    ],
    "Diabetes R²": [
        diabetes_r2,
        diabetes_poly_r2,
        diabetes_inter_r2
    ],
    "Diabetes RMSE": [
        diabetes_rmse,
        diabetes_poly_rmse,
        diabetes_inter_rmse
    ],
    "Alzheimer's R²": [
        alz_r2,
        alz_poly_r2,
        alz_inter_r2
    ],
    "Alzheimer's RMSE": [
        alz_rmse,
        alz_poly_rmse,
        alz_inter_rmse
    ]
})

results

,Model,Diabetes R²,Diabetes RMSE,Alzheimer's R²,Alzheimer's RMSE
0,Baseline,0.173310,0.632261,0.408240,0.368290
1,Polynomial,0.177848,0.630523,0.426099,0.362690
2,Interaction,0.184115,0.628115,0.425956,0.362735
